# 01 — Preparación de datos

Filtramos el DENUE para quedarnos solo con **escuelas en la Ciudad de México**
de los niveles de interés (preescolar, primaria, secundaria general, media
superior y media técnica terminal — sector público y privado).

Reproyectamos a UTM 14N (EPSG:32614) para tener coordenadas en metros.


In [ ]:
import unicodedata
from pathlib import Path
import numpy as np
import pandas as pd
from pyproj import Transformer

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
CSV = ROOT / 'denue_inegi_61_.csv'
OUT = ROOT / 'data' / 'processed' / 'escuelas_cdmx.parquet'
OUT.parent.mkdir(parents=True, exist_ok=True)
print('csv:', CSV.exists(), CSV)


In [ ]:
COLS = ['id','nom_estab','codigo_act','nombre_act','per_ocu','cve_ent','entidad',
        'cve_mun','municipio','latitud','longitud','fecha_alta']
df = pd.read_csv(CSV, encoding='latin-1', low_memory=False, usecols=COLS)
print('filas totales:', len(df))
df.head(2)


In [ ]:
# Filtro CDMX
df_cdmx = df[df['entidad'].str.contains('iudad de M', na=False)].copy()
print('CDMX:', len(df_cdmx))


In [ ]:
# Definición de las 12 categorías objetivo (claves SCIAN)
# 611111 preescolar publ, 611112 preescolar priv
# 611121 primaria publ,   611122 primaria priv  (cuidado: revisar abajo)
# Mejor usar el texto exacto del DENUE (con codificación latin-1 ya decodificada).
target_acts = [
    'Escuelas de educación preescolar del sector público',
    'Escuelas de educación preescolar del sector privado',
    'Escuelas de educación primaria del sector público',
    'Escuelas de educación primaria del sector privado',
    'Escuelas de educación secundaria general del sector público',
    'Escuelas de educación secundaria general del sector privado',
    'Escuelas de educación media superior del sector público',
    'Escuelas de educación media superior del sector privado',
    'Escuelas de educación media técnica terminal del sector público',
    'Escuelas de educación media técnica terminal del sector privado',
]

df_esc = df_cdmx[df_cdmx['nombre_act'].isin(target_acts)].copy()
print('escuelas objetivo en CDMX:', len(df_esc))
print(df_esc['nombre_act'].value_counts())


In [ ]:
# Validar lat/lon dentro del bounding box de CDMX
mask_geo = (
    df_esc['latitud'].between(19.0, 19.7) &
    df_esc['longitud'].between(-99.4, -98.9)
)
print('descartadas por lat/lon fuera de CDMX:', (~mask_geo).sum())
df_esc = df_esc[mask_geo].copy()


In [ ]:
# Derivar columnas 'nivel' y 'sector'
def parse_nivel(s: str) -> str:
    s = s.lower()
    if 'preescolar' in s: return 'preescolar'
    if 'primaria' in s: return 'primaria'
    if 'secundaria' in s: return 'secundaria'
    if 'media superior' in s: return 'media_superior'
    if 'media técnica' in s or 'media tecnica' in s: return 'media_tecnica'
    return 'otro'

def parse_sector(s: str) -> str:
    return 'privado' if 'privado' in s.lower() else 'público'

df_esc['nivel'] = df_esc['nombre_act'].map(parse_nivel)
df_esc['sector'] = df_esc['nombre_act'].map(parse_sector)
print(df_esc.groupby(['nivel','sector']).size().unstack(fill_value=0))


In [ ]:
# Reproyección a UTM 14N (EPSG:32614) — coordenadas en metros
tr = Transformer.from_crs(4326, 32614, always_xy=True)
x, y = tr.transform(df_esc['longitud'].values, df_esc['latitud'].values)
df_esc['x_utm'] = x
df_esc['y_utm'] = y
df_esc[['latitud','longitud','x_utm','y_utm']].describe()


In [ ]:
# Guardar
df_esc.reset_index(drop=True).to_parquet(OUT, index=False)
print('guardado:', OUT, '|', len(df_esc), 'filas')


## Verificación

- Esperamos ~5000–6000 escuelas tras el filtro.
- Cada nivel debe estar presente; preescolar y primaria dominan.
- Las coordenadas UTM x deben estar en el rango ~470000–510000, y en ~2120000–2170000.
